In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import h5py
import anndata as ad
from pathlib import Path

In [2]:
# read preprocessed log1p transformed data with 1000 most variable genes and selected genes from the paper
data_dir = '../data/frangieh'
adata = sc.read_h5ad(f"{data_dir}/dataset_hv_tim.h5ad")

### Compute log2FC

In [3]:
def split_data(adata, condition, pert_min_obs=50):
    data = adata[adata.obs['perturbation_2'].eq(condition)].copy()
    counts = data.obs["perturbation"].value_counts()
    keep = counts.index[counts >= pert_min_obs]
    return data[data.obs["perturbation"].isin(keep)].copy()

adata_ctl = split_data(adata, 'control')
adata_IFN = split_data(adata, 'IFNγ')
adata_co = split_data(adata, 'Co-culture')

In [4]:
def get_rank_genes_groups(adata, file_prefix):
    file = Path(data_dir) / f'{file_prefix}_ranked_tim.h5ad'
    if file.is_file():
        print(f"Loading ranked genes from {file.absolute()}")
        data = sc.read_h5ad(file)
        return data
    else:
        print(f"Computing ranked genes and saving to {file.absolute()}")
        data = adata.copy()
        sc.tl.rank_genes_groups(data, 'perturbation', reference='control', method='wilcoxon')
        data.write(file)
        return data

In [62]:
adata_ctl = get_rank_genes_groups(adata_ctl, 'adata_ctl')
adata_IFN = get_rank_genes_groups(adata_IFN, 'adata_IFN')
adata_co = get_rank_genes_groups(adata_co, 'adata_co')

Loading ranked genes from /tmp/pycharm_project_c95b1eb4/notebooks/../data/frangieh/adata_ctl_ranked_tim.h5ad
Loading ranked genes from /tmp/pycharm_project_c95b1eb4/notebooks/../data/frangieh/adata_IFN_ranked_tim.h5ad
Loading ranked genes from /tmp/pycharm_project_c95b1eb4/notebooks/../data/frangieh/adata_co_ranked_tim.h5ad


In [63]:
# df with cols groups:perturbation, names:gene, logfoldchange:log2FC perturbation vs. control
adata_ctl_FC = sc.get.rank_genes_groups_df(adata_ctl, None)
adata_IFN_FC = sc.get.rank_genes_groups_df(adata_IFN, None)
adata_co_FC = sc.get.rank_genes_groups_df(adata_co, None)

## Select 50 perturbations for modeling:

Perturbations that should be included:
- IFNGR1/2, JAK1/2, STAT1 since they are the main affected pathway by treatment conditions and cluster very separately
- HLA-B, (GBP1, WARS, CD74,) PFN1, (IDO1) since they were found in task 1 to be linked to different treatment conditions
- CD58 since it is the main finding of the original paper

Fill up the rest (42) with the perturbations with the most differentially expressed genes -> limitation: this introduces bias and increases model performance but cannot generalize to perturbations which have a weaker effect on gene expression.

In [69]:
perturbation_list = [
    'IFNGR1', 'IFNGR2', 'JAK1', 'JAK2', 'STAT1', 'HLA-B', 'PFN1', 'CD58',
] #left out due to obs count: 'GBP1', 'WARS', 'CD74', 'IDO1'

# select 38 perturbations with highest number of differentially expressed genes in co-culture condition.
# to consider a gene differentially expressed, use log2FC cutoff of 2
log2fc_cutoff = 2.0

de_genes = adata_co_FC[adata_co_FC["logfoldchanges"].abs() >= log2fc_cutoff]
counts = de_genes.groupby("group")["names"].nunique().sort_values(ascending=False)

keep = counts[~counts.index.isin(perturbation_list)].head(42).index
perturbation_list = perturbation_list + keep.astype('str').tolist()

adata_ctl = adata_ctl[adata_ctl.obs['perturbation'].isin(perturbation_list),:].copy()
adata_IFN = adata_IFN[adata_IFN.obs['perturbation'].isin(perturbation_list),:].copy()
adata_co = adata_co[adata_co.obs['perturbation'].isin(perturbation_list),:].copy()